# Classificação de radiografias de tórax — Parte 2: aumento de dados

Este é o segundo notebook do projeto. Retoma a CNN de referência e explora preparação e aumento de dados, bem como pesos de classe, para estudar o efeito dessas escolhas na classificação. Execute-o a partir da raiz do projeto, depois da Parte 1, para manter os resultados organizados em `results/`. As comparações com CSV de experiências históricas são opcionais.


## 1. Imports

In [ ]:
# !pip install -U tensorflow numpy pandas matplotlib

import tensorflow as tf
from tensorflow.keras import layers, metrics, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
import random
import numpy as np
import os 
import pandas as pd
import matplotlib.pyplot as plt
import os


## 2. Data

### 2.1 Data Load

In [ ]:
# Caminhos dos dados de treino e teste
train_dir = 'data/train'
test_dir = 'data/test'

# Parâmetros principais
batch_size = 32
img_size = (256, 256)

# Carregar conjunto de treino (labels inferidas pelas pastas)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',  
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    color_mode='grayscale',
)

# Carregar conjunto de teste (sem baralhar)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
    color_mode='grayscale'
)

# Normalizar valores de píxeis para [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)

# Aplicar normalização aos datasets
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

### 2.2 Data Analysis

- Class Normal: 0
- Class Pneumonia: 1

Observe que existe um desbalanceamento entre as classes em ambos os datasets, treino e teste, para este primeiro trabalho iremos realizar uma redução no numero de exemplos da class majoritaria.

In [ ]:
# Função para contar o número de exemplos por classe num dataset
def count_exemples_classe(dataset):
    counts = {}

    for _, labels in dataset:
        labels_np = labels.numpy()
        for label in labels_np:
            label = int(label)
            if label in counts:
                counts[label] += 1
            else:
                counts[label] = 1
    
    return counts

In [ ]:
# Contar exemplos nas classes de treino e teste
train_count = count_exemples_classe(train_dataset)
test_count = count_exemples_classe(test_dataset)

# Ordenar resultados por classe
train_count = dict(sorted(train_count.items()))
test_count = dict(sorted(test_count.items()))

print("Train dataset class counts:", train_count)
print("Test dataset class counts:", test_count)

In [ ]:
# Função para calcular diferença entre classes (para verificar desbalanceamento)
def count_difference(counts):

    if not counts:
        return {}
    
    min_count = min(counts.values())
    diff_counts = {cls: count - min_count for cls, count in counts.items()}
    return diff_counts

In [ ]:
# Calcular diferenças entre classes
train_diff = count_difference(train_count)
test_diff = count_difference(test_count)

print("Train difference:", train_diff)
print("Test difference:", test_diff)

### 2.3 Data Preparation

In [ ]:
def undersample_dataset(dataset, batch_size=32, seed=42):

    # Como o dataset é composto por batches, precisamos concatenar tudo
    # para obter arrays únicos de imagens e respetivas labels.
    images = []
    labels = []
    for batch_images, batch_labels in dataset:
        images.append(batch_images.numpy())
        labels.append(batch_labels.numpy())
    
    # Junta todos os batches num único array
    images = np.concatenate(images, axis=0)
    labels = np.concatenate(labels, axis=0)

    # Separar índices das classes
    class0_indices = np.where(labels == 0)[0]
    class1_indices = np.where(labels == 1)[0]

    # Determinar o número mínimo de exemplos entre as classes
    min_count = min(len(class0_indices), len(class1_indices))

    # Baralhamos os índices para garantir aleatoriedade
    rng = np.random.default_rng(seed)
    rng.shuffle(class0_indices)
    rng.shuffle(class1_indices)

    # Selecionamos o mesmo número de exemplos de cada classe
    selected_indices = np.concatenate([
        class0_indices[:min_count],
        class1_indices[:min_count]
    ])
    rng.shuffle(selected_indices)

    # Usamos apenas as imagens e labels selecionadas
    balanced_images = images[selected_indices]
    balanced_labels = labels[selected_indices]

    # Convertemos novamente para um tf.data.Dataset e agrupamos em batches
    balanced_dataset = tf.data.Dataset.from_tensor_slices((balanced_images, balanced_labels))
    balanced_dataset = balanced_dataset.batch(batch_size)
    return balanced_dataset


In [ ]:
'''
# Aplica a função ao conjunto de treino
balanced_train_dataset = undersample_dataset(train_dataset, batch_size=batch_size)
balanced_train_count = count_exemples_classe(balanced_train_dataset)
print("Balanced Train dataset class counts:", dict(sorted(balanced_train_count.items())))

#Aplicação opcional ao conjunto de teste (opcional)
balanced_test_dataset = undersample_dataset(test_dataset, batch_size=batch_size)
balanced_test_count = count_exemples_classe(balanced_test_dataset)
print("Balanced Test dataset class counts:", dict(sorted(balanced_test_count.items())))
'''


## Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),             # Rotação aleatória ±10%
    layers.RandomZoom(0.1),                 # Zoom aleatório
    layers.RandomTranslation(0.1, 0.1),     # Translação aleatória
])

augmented_train_dataset = train_dataset.map(        # balanced_train_dataset se quiser aplicar undersample
    lambda x, y: (data_augmentation(x, training=True), y)
)

## 3. Model

In [ ]:
# Modelo CNN
model = tf.keras.models.Sequential([

    # camada convolucional + Pooling
    layers.Conv2D(16, 5, padding='same', activation='relu', input_shape=(img_size[0], img_size[1], 1)),
    layers.MaxPooling2D(pool_size=(2,2), strides=1, padding='same'),

    # camada convolucional + Pooling
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

    # camada convolucional + Pooling
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

    layers.Dropout(0.4),    # Evita overfitting
    
    layers.Flatten(),       # Achata o output para uma dimensão
 
    layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)), # Camada densa
 
    layers.Dense(1, activation="sigmoid") # Saída binária (0 ou 1)
])

# Early stopping para parar o treino quando a validação não melhora
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',      
    patience=3,              
    restore_best_weights=True
)

In [ ]:
# Compilação do modelo
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',                             # accuracy padrão
        metrics.Precision(name='precision'),    # precisão = TP / (TP + FP)
        metrics.Recall(name='recall'),          # recall = TP / (TP + FN)
        metrics.AUC(name='auc'),                # AUC da curva ROC
        metrics.TruePositives(name='tp'),       # contagem de verdadeiros positivos
        metrics.TrueNegatives(name='tn'),       # contagem de verdadeiros negativos
        metrics.FalsePositives(name='fp'),      # contagem de falsos positivos
        metrics.FalseNegatives(name='fn')       # contagem de falsos negativos
    ]
)
# N = total de amostras / (n_classes = número de classes . n_i = nº de imagens da classe i)
class_weights = {0: 1.94, 1: 0.67}

# Treinar o modelo com early stopping
history = model.fit(
    augmented_train_dataset,
    validation_data=test_dataset,
    epochs=100,             
    callbacks=[early_stop],
    class_weight=class_weights
)

# Guardar histórico do treino
os.makedirs('results', exist_ok=True)

df_history = pd.DataFrame(history.history)
csv_path = os.path.join('results', 'model_data_augmentation.csv')
df_history.to_csv(csv_path, index=False)

print(f"Histórico guardado em '{csv_path}'.")

##  4. History

In [ ]:
def analisar_modelo(csv_path):
    # Ler CSV
    df = pd.read_csv(csv_path)
    
    # Lista de métricas principais
    metrics = ['accuracy', 'loss', 'precision', 'recall', 'auc']
    
    # Criar gráficos separadamente
    for metric in metrics:
        val_metric = f'val_{metric}'
        
        if metric in df.columns and val_metric in df.columns:
            plt.figure(figsize=(8, 5))
            plt.plot(df[metric], label=f'Train {metric}', marker='o')
            plt.plot(df[val_metric], label=f'Validation {metric}', marker='s')
            plt.title(f'{metric.upper()} por Epoch')
            plt.xlabel('Epoch')
            plt.ylabel(metric.capitalize())
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)

            # Limitar eixo Y
            if metric == 'loss':
                plt.ylim(0, max(df[metric].max(), df[val_metric].max()) * 1.1)
            else:
                plt.ylim(0, 1.05)

            plt.tight_layout()
            plt.show()
    
    # Última epoch
    ultima_epoch = df.iloc[-1]

    # Lista de todas as métricas possíveis
    todas_metricas = [
        'accuracy','auc','fn','fp','loss','precision','recall','tn','tp',
        'val_accuracy','val_auc','val_fn','val_fp','val_loss','val_precision','val_recall','val_tn','val_tp'
    ]

    # Criar lista de tuplas (métrica, valor) apenas se a métrica existir no CSV
    tabela_valores = [
        (m, round(ultima_epoch[m], 4)) 
        for m in todas_metricas if m in ultima_epoch
    ]

    # Converter para DataFrame
    ultima_epoch_df = pd.DataFrame(tabela_valores, columns=['Métrica', 'Valor'])

    print(f"\nValores da Última Epoch ({csv_path}):")
    display(ultima_epoch_df)


### Variante histórica 1

CNN inicial sem redução espacial suficiente. Não é treinada pelas células atuais.


### Variante histórica 2

CNN com redução espacial nas últimas camadas de pooling. A visualização do CSV histórico é opcional; este modelo não é treinado pelas células atuais.


In [ ]:
csv_historico = "results/Model2.csv"
if os.path.isfile(csv_historico):
    analisar_modelo(csv_historico)
else:
    print(f"[INFO] Resultado histórico não disponível: {csv_historico}")


### Variante histórica 3

Abordagem em escala de cinzentos, com normalização e subamostragem. O CSV histórico não acompanha o projeto.


In [ ]:
csv_historico = "results/Model3.csv"
if os.path.isfile(csv_historico):
    analisar_modelo(csv_historico)
else:
    print(f"[INFO] Resultado histórico não disponível: {csv_historico}")


### Variante histórica 4

CNN com dropout e regularização L2. O CSV histórico pode ser consultado quando estiver disponível, mas esta variante não é treinada aqui.


In [ ]:
csv_historico = "results/Model4.csv"
if os.path.isfile(csv_historico):
    analisar_modelo(csv_historico)
else:
    print(f"[INFO] Resultado histórico não disponível: {csv_historico}")


### Modelo executado neste notebook

Este percurso aplica aumento de dados e pesos de classe à CNN definida nas células anteriores. O histórico é guardado em `results/model_data_augmentation.csv`. A comparação com variantes históricas depende dos respetivos CSV, que não fazem parte deste projeto.


In [ ]:
csv_historico = "results/Model5.csv"
if os.path.isfile(csv_historico):
    analisar_modelo(csv_historico)
else:
    print(f"[INFO] Resultado histórico não disponível: {csv_historico}")


In [ ]:
def ultima_epoch(pasta_results):

    # Lista todos os arquivos CSV na pasta
    arquivos = [f for f in os.listdir(pasta_results) if f.endswith('.csv')]
    
    tabela_final = pd.DataFrame()
    
    for arquivo in arquivos:
        caminho_csv = os.path.join(pasta_results, arquivo)
        df = pd.read_csv(caminho_csv)
        
        # Última epoch
        ultima_epoch = df.iloc[-1]
        
        # Lista de todas as métricas existentes no CSV
        todas_metricas = [
            'accuracy','auc','fn','fp','loss','precision','recall','tn','tp',
            'val_accuracy','val_auc','val_fn','val_fp','val_loss','val_precision','val_recall','val_tn','val_tp'
        ]
        
        # Criar série só com métricas existentes
        serie_modelo = pd.Series(
            {m: round(ultima_epoch[m], 4) for m in todas_metricas if m in ultima_epoch},
            name=os.path.splitext(arquivo)[0]  # nome do modelo
        )
        
        # Concatenar ao DataFrame final
        tabela_final = pd.concat([tabela_final, serie_modelo], axis=1)
    
    # Mostrar a tabela
    print("Valores da Última Epoch de todos os modelos:")
    display(tabela_final)
    
    return tabela_final

tabela_modelos = ultima_epoch('results')

## Conclusão e limites


Este notebook treina uma CNN com aumento de dados e pesos de classe e grava o histórico em `results/model_data_augmentation.csv`. As descrições das variantes anteriores são contexto histórico e não correspondem a treinos executados pelas células atuais. O conjunto `test_dataset` é usado como validação durante o treino; as métricas obtidas não representam uma avaliação final independente. Não se pode concluir adequação para uso clínico a partir desta experiência académica.
